# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SaaDasim05/Flyrank-ML-Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

My lane is Refresh / Content Opportunity Scoring. The goal is to identify content items that may benefit from review or updating, using measurable search-performance signals. The output would be a ranked review queue rather than an automatic decision to change content. This fits my earlier work because I observed that a large proportion of content items were trending down, making content refresh a useful decision-support problem.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

- One row represents one content item for a client on a specific reporting date.

- I will use the daily content-performance fact table, fact_content_daily_performance, and supporting content/query tables only when required for additional features.

- I will use the March 2026 panel as the main verification window, because it is a mid-panel month and avoids using the final June 2026 month as an outcome/test period.

- I will rank content items according to their likelihood of being useful candidates for review or refresh. A possible proxy is whether impressions decline substantially over a future comparison window.

- I will deliberately exclude trend_direction and trend_pct from predictive features when the target is derived from them, because using them would leak information from the outcome into the model.



In [13]:
import os
import duckdb

from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError("HF_TOKEN not found in Colab Secrets.")

con = duckdb.connect()

con.execute(f"""
    CREATE OR REPLACE SECRET hf (
        TYPE HUGGINGFACE,
        TOKEN '{HF_TOKEN}'
    )
""")

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_clients":
        f"read_parquet('{REL}/dim_clients.parquet')",

    "dim_content":
        f"read_parquet('{REL}/dim_content.parquet')",

    "fact_daily":
        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",

    "fact_query_90d":
        f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

print("DuckDB connected.")

DuckDB connected.


In [14]:
con = duckdb.connect()

con.execute(f"""
    CREATE OR REPLACE SECRET hf (
        TYPE HUGGINGFACE,
        TOKEN '{HF_TOKEN}'
    )
""")

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_clients":
        f"read_parquet('{REL}/dim_clients.parquet')",

    "dim_content":
        f"read_parquet('{REL}/dim_content.parquet')",

    "fact_daily":
        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",

    "fact_query_90d":
        f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

print("DuckDB connected.")

DuckDB connected.


In [15]:
test = con.sql(f"""
    SELECT *
    FROM {TABLES['dim_clients']}
    LIMIT 5
""").df()

test

,client_hash_id,is_active,has_gsc_access,has_ga4_access,access_profile,client_created_date,client_updated_date,gsc_data_start,ga4_data_start
0,client_04660893ae39614a,True,True,True,gsc_and_ga4,2026-04-15,2026-06-27,NaT,2026-05-22
1,client_05475c07ed21a83a,True,False,False,no_search_or_analytics_access,2026-04-01,2026-06-27,NaT,NaT
2,client_06d356715a8ff3b6,True,True,True,gsc_and_ga4,2026-03-23,2026-07-05,2026-04-10,2026-04-06
3,client_0797ff3a1fc9a6a5,True,False,False,no_search_or_analytics_access,2025-05-26,2026-06-27,2025-11-05,NaT
4,client_08a6a72ff48e62c0,True,True,False,gsc_only,2025-05-26,2026-06-27,2025-09-24,NaT


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [17]:
# Show the columns without reading the whole table
schema = con.sql(f"""
    SELECT *
    FROM {TABLES['fact_daily']}
    LIMIT 0
""").df()

print(schema.columns.tolist())

['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


In [18]:
grain_check = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,
        COUNT(*) AS row_count
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 10
""").df()

print("Duplicate client × content × date combinations:", len(grain_check))
grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate client × content × date combinations: 0


,client_hash_id,content_hash_id,report_date,row_count


In [16]:
date_check = con.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS start_date,
        MAX(report_date) AS end_date
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'
""").df()

date_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,row_count,start_date,end_date
0,9841378,2026-03-01,2026-03-31


In [19]:
availability_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS gsc_available_rows
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'
""").df()

availability_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_available_rows
0,9841378,3611061


In [20]:
availability_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS gsc_available_rows
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'
""").df()

availability_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_available_rows
0,9841378,3611061


In [21]:
feature_frame = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS impressions,
        SUM(gsc_clicks) AS clicks,

        AVG(
            CASE
                WHEN gsc_data_available IS TRUE
                THEN gsc_avg_position
            END
        ) AS avg_position,

        SUM(sessions_organic) AS organic_sessions,

        COUNT(DISTINCT report_date) FILTER (
            WHERE gsc_impressions > 0
        ) AS days_with_impressions

    FROM {TABLES['fact_daily']}

    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'

    GROUP BY 1, 2
""").df()

print(f"Rows in feature frame: {len(feature_frame):,}")

feature_frame.head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows in feature frame: 331,437


,client_hash_id,content_hash_id,impressions,clicks,avg_position,organic_sessions,days_with_impressions
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,6523.0,7.0,7.209549,1.0,31
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,453.0,0.0,2.987198,0.0,31
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,5630.0,6.0,6.724039,6.0,31
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,4944.0,13.0,7.244844,1.0,31
4,client_73cda7b4e4f265ea,content_f39be42b42a4e8f6,42.0,0.0,14.432540,0.0,21
5,client_73cda7b4e4f265ea,content_1855a661b4d36130,429.0,1.0,4.209227,0.0,31
6,client_73cda7b4e4f265ea,content_5d412fba6e1a2582,223.0,1.0,9.445635,0.0,31
7,client_73cda7b4e4f265ea,content_1f380a642aed423b,96.0,1.0,6.014516,0.0,31
8,client_73cda7b4e4f265ea,content_22c063002b7c1caf,314.0,1.0,9.155335,0.0,31
9,client_73cda7b4e4f265ea,content_aafb2ab7e5fc80d0,7709.0,20.0,5.258331,4.0,31


In [22]:
availability_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS gsc_available_rows
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'
""").df()

availability_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_available_rows
0,9841378,3611061


In [23]:
feature_frame = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS impressions,
        SUM(gsc_clicks) AS clicks,

        AVG(
            CASE
                WHEN gsc_data_available IS TRUE
                THEN gsc_avg_position
            END
        ) AS avg_position,

        SUM(sessions_organic) AS organic_sessions,

        COUNT(DISTINCT report_date) FILTER (
            WHERE gsc_impressions > 0
        ) AS days_with_impressions

    FROM {TABLES['fact_daily']}

    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'

    GROUP BY 1, 2
""").df()

print(f"Feature rows: {len(feature_frame):,}")

feature_frame.head(10)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature rows: 331,437


,client_hash_id,content_hash_id,impressions,clicks,avg_position,organic_sessions,days_with_impressions
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,1140.0,2.0,4.394234,0.0,31
1,client_73cda7b4e4f265ea,content_05597932fe4da067,57.0,0.0,2.714744,0.0,26
2,client_73cda7b4e4f265ea,content_905aa32a0230694e,149.0,0.0,6.481453,0.0,30
3,client_73cda7b4e4f265ea,content_05434271b257bb68,1421.0,6.0,6.320337,4.0,31
4,client_73cda7b4e4f265ea,content_d056587ff7faca0c,2770.0,16.0,4.459107,0.0,31
5,client_73cda7b4e4f265ea,content_bfd1e41c2af250c8,48.0,0.0,14.753175,0.0,21
6,client_73cda7b4e4f265ea,content_2662845f598544ef,150.0,1.0,6.341880,0.0,30
7,client_73cda7b4e4f265ea,content_22610b0934f8825e,67.0,0.0,12.791667,0.0,26
8,client_73cda7b4e4f265ea,content_712c365258cee05c,6048.0,23.0,4.950311,6.0,31
9,client_73cda7b4e4f265ea,content_476c37c366920c1b,223.0,0.0,50.390299,0.0,30


### Five features and when they are available

- **impressions** — Knowable at the decision moment because it aggregates GSC impressions observed during the March feature window.
- **clicks** — Knowable at the decision moment because it aggregates historical GSC clicks observed during the feature window.
- **avg_position** — Knowable at the decision moment because it summarizes previously observed GSC average-position measurements.
- **organic_sessions** — Knowable at the decision moment because it aggregates historical organic sessions observed during the feature window.
- **days_with_impressions** — Knowable at the decision moment because it counts the historical days on which the content received GSC impressions.

The decision moment for this exercise is after the feature window has closed. These features are therefore historical measurements available at that point, not future information.

In [24]:
import numpy as np

march_label = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(
            CASE
                WHEN report_date < DATE '2026-03-16'
                THEN gsc_impressions
                ELSE 0
            END
        ) AS impressions_first15,

        SUM(
            CASE
                WHEN report_date >= DATE '2026-03-16'
                THEN gsc_impressions
                ELSE 0
            END
        ) AS impressions_last16

    FROM {TABLES['fact_daily']}

    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'

    GROUP BY 1, 2
""").df()

march_label["is_declining"] = (
    march_label["impressions_last16"]
    < 0.8 * march_label["impressions_first15"]
).astype(int)

experiment = feature_frame.merge(
    march_label[
        ["client_hash_id", "content_hash_id", "is_declining"]
    ],
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

print(f"Experiment rows: {len(experiment):,}")
print(f"Decline rate: {experiment['is_declining'].mean():.3f}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Experiment rows: 331,437
Decline rate: 0.150


In [25]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

honest_features = [
    "impressions",
    "clicks",
    "avg_position",
    "organic_sessions",
    "days_with_impressions"
]

model_df = experiment.dropna(
    subset=honest_features + ["is_declining"]
).copy()

X_honest = model_df[honest_features]
y = model_df["is_declining"]

X_train, X_test, y_train, y_test = train_test_split(
    X_honest,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

honest_model = DecisionTreeClassifier(
    max_depth=4,
    random_state=42
)

honest_model.fit(X_train, y_train)

honest_pred = honest_model.predict(X_test)

honest_score = accuracy_score(
    y_test,
    honest_pred
)

print(f"Honest model accuracy: {honest_score:.3f}")

Honest model accuracy: 0.723


In [26]:
# Deliberate leakage experiment:
# give the model the answer itself.

leaky_features = honest_features + ["is_declining"]

X_leaky = model_df[leaky_features]

X_train, X_test, y_train, y_test = train_test_split(
    X_leaky,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

leaky_model = DecisionTreeClassifier(
    max_depth=4,
    random_state=42
)

leaky_model.fit(X_train, y_train)

leaky_pred = leaky_model.predict(X_test)

leaky_score = accuracy_score(
    y_test,
    leaky_pred
)

print(f"Leaky model accuracy: {leaky_score:.3f}")

Leaky model accuracy: 1.000


### The leakage trap

I deliberately added `is_declining`, the target variable, to the feature set. The model then achieved **1.000 accuracy**, compared with **0.723 accuracy** for the honest model.

This apparent improvement is not a real modeling improvement. It happened because the leaky feature directly contains the outcome the model is supposed to predict. The 1.000 result is therefore invalid and must not be used as evidence of model quality.

I removed `is_declining` from the final feature set and kept the **0.723 honest accuracy** as the legitimate result of this simple experiment.

This demonstrates why every feature must be available at the real decision moment and must not be derived from the target or future outcome.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Unbalanced client history

Client histories do not all have the same start dates or data availability. Therefore, the March 2026 slice can contain different amounts and types of historical coverage across clients. This may affect comparability between content items and should be considered when building later models and validation splits.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.